# Recursive IICA Chain — the chained unified model

Mirrors `03_recursive_iica_chain.ipynb`.  A [UM] node is the compound IICA
morphism `encodings -> HLLSets -> encodings`.  Chaining nodes composes the
morphism (Principle 2: composition of IICA is IICA).  The chain runs on
quantized V-JEPA tid streams — the same streams the encoder boundary emits.

In [1]:
import sys
import pathlib

ROOT = pathlib.Path.cwd()
if not (ROOT / "ewm_jepa").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import torch
from ewm_jepa import Quantizer
import hllset_py
from hllset_cortex import HLLSetFilter
from hllset_cortex.domain import encoding_tokenizer, hllset_from_ids, tid

# A deterministic tid-stream source: the same quantizer the JEPA pipeline
# uses, fed with synthetic encodings.  The EWM boundary is encoding-agnostic —
# these tids are interchangeable with V-JEPA quantized patch encodings.
def make_tid_stream(n=200, dim=1024, codebook_size=2048, seed=0):
    q = Quantizer(dim=dim, codebook_size=codebook_size, device="cpu", seed=seed)
    # decorrelate the data seed from the codebook seed so streams differ
    z = torch.randn(n, dim, generator=torch.Generator().manual_seed(1000 + seed))
    return q.ids_to_text(q.encode(z))


## 1. The building block

A chain node is just `HLLSetFilter` with a gate: ingest → gate ∩ → LUT →
materialize.  The output stream feeds the next node.

In [2]:
def make_node(gate_ids):
    f = HLLSetFilter()
    f.tokenizer = encoding_tokenizer()
    f.gate_hllset = hllset_from_ids(gate_ids)
    return f

def node_pass(node, text):
    res = node.process_text(text)
    # unigrams only, in materialized order
    return " ".join(t for t in res.token_strings if "\x00" not in t)

stream = make_tid_stream(n=40, seed=3)
node = make_node(range(2048))
out = node_pass(node, stream)
print("in :", stream)
print("out:", out)


in : tid1334 tid191 tid1133 tid1584 tid1797 tid1805 tid1760 tid372 tid848 tid1294 tid673 tid1177 tid875 tid1962 tid1977 tid497 tid603 tid1698 tid1452 tid1588 tid164 tid155 tid469 tid216 tid901 tid1984 tid259 tid92 tid941 tid800 tid18 tid1738 tid369 tid1992 tid1418 tid1527 tid485 tid1069 tid1808 tid1907
out: tid155 tid1133 tid1738 tid1805 tid673 tid164 tid216 tid1527 tid1760 tid1069 tid1334 tid469 tid800 tid1962 tid191 tid941 tid1808 tid1698 tid259 tid603 tid1294 tid1797 tid1177 tid485 tid848 tid1907 tid369 tid1584 tid901 tid1977 tid92 tid18 tid497 tid1418 tid372 tid875 tid1452 tid1984 tid1992 tid1588


## 2. IICA chain verification

Three nodes in a row.  IICA composition predicts: same input → same output,
every run, and the chain converges to a fixed point after one full pass.

In [3]:
chain = [make_node(range(2048)) for _ in range(3)]
x = stream
for i, node in enumerate(chain):
    x = node_pass(node, x)
    print(f"after node {i}: {len(x.split())} tids")

x2 = stream
for node in chain:
    x2 = node_pass(node, x2)
print("deterministic:", x == x2)


after node 0: 40 tids
after node 1: 39 tids
after node 2: 39 tids
deterministic: True


## 3. Progressive gate chain

Each node narrows the gate: node 1 admits 2048 tids, node 2 admits 1024,
node 3 admits 512.  The chain filters the stream to the intersection of all
world views — a structural "cortical brake" cascade.

In [4]:
gates = [range(2048), range(1024), range(512)]
x = stream
for i, g in enumerate(gates):
    x = node_pass(make_node(g), x)
    print(f"gate {i} ({len(list(g))} tids): {len(x.split())} tids survive")


gate 0 (2048 tids): 40 tids survive
gate 1 (1024 tids): 23 tids survive
gate 2 (512 tids): 15 tids survive


## 4. Feedback convergence

Close the loop: the output feeds back as input.  The chain reaches a fixed
point because HLLSet union is idempotent and the LUT is monotonic — the
Noether union invariant in miniature.

In [5]:
node = make_node(range(2048))
x = stream
for step in range(5):
    x_next = node_pass(node, x)
    print(f"step {step}: {len(x.split())} tids")
    if x_next == x:
        print("fixed point reached")
        break
    x = x_next


step 0: 40 tids
step 1: 40 tids
fixed point reached


## 5. Summary

`[E] -> chain -> [E]` holds for V-JEPA encodings exactly as it held for
DeepSeek-OCR encodings: the EWM boundary is encoding-agnostic, and IICA
composition means the chain needs no coordination, no versioning, and no
naming at any depth.